In [4]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import mlflow
from mlflow import log_metric, log_param, log_artifact

## Load models

In [5]:
from pathlib import Path

# Get all models with endswith .pkl and .keras from pipelines/models directory
import os
base_dir = Path.cwd().resolve().parents[3]
model_dir = base_dir / "pipelines" / "models"
model_files = [f for f in os.listdir(model_dir) if f.endswith(".pkl") or f.endswith(".keras")]
print("Model files found:", model_files)

Model files found: ['lstm_model.keras', 'scaler_X.pkl', 'scaler_profit.pkl', 'scaler_sales.pkl']


## Connect Qdrant Cloud + Init Collections

In [9]:
import os
from pathlib import Path
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Configuration
ENV_PATH = Path.cwd().resolve().parents[2] / ".env"
load_dotenv(dotenv_path=ENV_PATH)

QDRANT_CLUSTER = os.getenv("Qdrant_Cluster_Endpoint")
QDRANT_API_KEY = os.getenv("Qdrant_API_KEY")
VECTOR_SIZE = 400

qdrant_client = QdrantClient(
    url=QDRANT_CLUSTER,
    api_key=QDRANT_API_KEY,
    prefer_grpc=False,
    timeout=60
)

display(Markdown(f"✅ **Qdrant cloud storage** at `{QDRANT_CLUSTER}`"))

try:
    collections = qdrant_client.get_collections().collections
    names = [c.name for c in collections]
    display(HTML(f"""
    <div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9'>
        ✅ <b>Connected to Qdrant Cloud</b><br>
        🌐 Cluster: <code>{QDRANT_CLUSTER}</code><br>
        📦 Existing collections: <code>{names if names else 'None yet'}</code>
    </div>
    """))
except Exception as e:
    display(HTML(f"<div style='color:red;padding:10px'>❌ Connection Failed: {e}</div>"))    

# ── CELL: Connect Qdrant Cloud + Init Collections ────────────
def init_collection(name: str) -> bool:
    """Returns True  → newly created  (needs indexing)
    Returns False → already exists (skip indexing)

    ✅ Also force re-index if collection EXISTS but is EMPTY
    """
    existing = [c.name for c in qdrant_client.get_collections().collections]

    if name in existing:
        count = qdrant_client.count(collection_name=name).count

        if count == 0:
            display(HTML(f"""
            <div style='border:1px solid #f59e0b;border-radius:8px;padding:10px;
                        background:#fffbeb;margin:4px 0'>
                ⚠️ <b>{name}</b> exists but has <b>0 vectors</b> — deleting and re-creating...
            </div>
            """))
            qdrant_client.delete_collection(collection_name=name)
            qdrant_client.create_collection(
                collection_name=name,
                vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
            )
            return True # needs indexing after re-create
        
        else:
            display(HTML(f"""
            <div style='border:1px solid #2196F3;border-radius:8px;padding:10px;
                        background:#f0f8ff;margin:4px 0'>
                ⚡ <b>{name}</b> already on Qdrant Cloud —
                <b>{count:,} vectors</b> — skipping re-index ✅
            </div>
            """))
            return False  # ✅ already indexed, skip

    else:
        qdrant_client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
        )
        display(HTML(f"""
        <div style='border:1px solid #4CAF50;border-radius:8px;padding:10px;background:#f9fff9;margin:4px 0'>
            ✅ <b>{name}</b> created (size={VECTOR_SIZE}, COSINE)
        </div>
        """))
        return True
    
display(Markdown("### 📦 Qdrant Cloud — Collection Init"))
need_index_products = init_collection("fashion_products")
need_index_pdf = init_collection("rag_report")

display(HTML(f"""
<div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9;margin-top:8px'>
    📋 <b>Index Status</b><br>
    📦 fashion_products : <b>{'🔄 Will index' if need_index_products else '✅ Ready'}</b><br>
    📄 rag_report       : <b>{'🔄 Will index' if need_index_pdf else '✅ Ready'}</b>
</div>
"""))

✅ **Qdrant cloud storage** at `https://38c48295-4cf5-4352-a5bb-6dd418900b84.eu-west-2-0.aws.cloud.qdrant.io`

### 📦 Qdrant Cloud — Collection Init

## LangChain HuggingFaceEmbeddings

In [11]:
import traceback
from IPython.display import display, HTML
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

# Define models
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "mps"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32}
)

# Test embedding
test_vec    = embeddings.embed_query("LLM fashion recommendation test")
VECTOR_SIZE = len(test_vec)

display(HTML(f"""
<div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9'>
    ✅ <b>HuggingFaceEmbeddings ready</b><br>
    🤗 Model       : <code>{EMBED_MODEL}</code><br>
    📐 Vector size : <b>{VECTOR_SIZE}</b><br>
    🔧 Via         : LangChain <code>HuggingFaceEmbeddings</code>
</div>
"""))

## Connect Qdrant Cloud + Init collections

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(
    url=QDRANT_CLUSTER,
    api_key=QDRANT_API_KEY,
    prefer_grpc=False,
    timeout=60
)

display(Markdown(f"✅ **Qdrant cloud storage** at `{QDRANT_CLUSTER}`"))

try:
    collections = qdrant_client.get_collections().collections
    names = [c.name for c in collections]
    display(HTML(f"""
    <div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9'>
        ✅ <b>Connected to Qdrant Cloud</b><br>
        🌐 Cluster: <code>{QDRANT_CLUSTER}</code><br>
        📦 Existing collections: <code>{names if names else 'None yet'}</code>
    </div>
    """))
except Exception as e:
    display(HTML(f"<div style='color:red;padding:10px'>❌ Connection Failed: {e}</div>"))    

# ── CELL: Connect Qdrant Cloud + Init Collections ────────────
def init_collection(name: str) -> bool:
    """Returns True  → newly created  (needs indexing)
    Returns False → already exists (skip indexing)

    ✅ Also force re-index if collection EXISTS but is EMPTY
    """
    existing = [c.name for c in qdrant_client.get_collections().collections]

    if name in existing:
        count = qdrant_client.count(collection_name=name).count

        if count == 0:
            display(HTML(f"""
            <div style='border:1px solid #f59e0b;border-radius:8px;padding:10px;
                        background:#fffbeb;margin:4px 0'>
                ⚠️ <b>{name}</b> exists but has <b>0 vectors</b> — deleting and re-creating...
            </div>
            """))
            qdrant_client.delete_collection(collection_name=name)
            qdrant_client.create_collection(
                collection_name=name,
                vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
            )
            return True # needs indexing after re-create
        
        else:
            display(HTML(f"""
            <div style='border:1px solid #2196F3;border-radius:8px;padding:10px;
                        background:#f0f8ff;margin:4px 0'>
                ⚡ <b>{name}</b> already on Qdrant Cloud —
                <b>{count:,} vectors</b> — skipping re-index ✅
            </div>
            """))
            return False  # ✅ already indexed, skip

    else:
        qdrant_client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
        )
        display(HTML(f"""
        <div style='border:1px solid #4CAF50;border-radius:8px;padding:10px;background:#f9fff9;margin:4px 0'>
            ✅ <b>{name}</b> created (size={VECTOR_SIZE}, COSINE)
        </div>
        """))
        return True
    
display(Markdown("### 📦 Qdrant Cloud — Collection Init"))
need_index_products = init_collection("fashion_products")
need_index_pdf = init_collection("rag_report")

display(HTML(f"""
<div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9;margin-top:8px'>
    📋 <b>Index Status</b><br>
    📦 fashion_products : <b>{'🔄 Will index' if need_index_products else '✅ Ready'}</b><br>
    📄 rag_report       : <b>{'🔄 Will index' if need_index_pdf else '✅ Ready'}</b>
</div>
"""))

✅ **Qdrant cloud storage** at `https://38c48295-4cf5-4352-a5bb-6dd418900b84.eu-west-2-0.aws.cloud.qdrant.io`

### 📦 Qdrant Cloud — Collection Init

## Langchain Qdrant VectorStore Setup

In [13]:
from langchain_qdrant import QdrantVectorStore

# config
COLLECTION = "fashion_products"
PDF_COLLECTION = "rag_report"

# Product vector store
product_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings
)

# PDF report vector store
pdf_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=PDF_COLLECTION,
    embedding=embeddings
)

# Verify vector counts
prod_count = qdrant_client.count(collection_name=COLLECTION).count
pdf_count = qdrant_client.count(collection_name=PDF_COLLECTION).count

display(HTML(f"""
<div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9'>
    ✅ <b>LangChain QdrantVectorStore ready</b><br>
    📦 Product store : <code>{COLLECTION}</code><br>
    📄 PDF store     : <code>{PDF_COLLECTION}</code><br>
    🔧 Embeddings    : <code>HuggingFaceEmbeddings ({EMBED_MODEL})</code>
</div>
"""))

In [17]:
# Load database parquet for index protudcts

import pandas as pd
df_merged = pd.read_parquet("../../../../pipelines/ai_engineer/LLM/fashion_recommendation")
df_merged

,item_id,date,purchase_count,view_count,price,stocks,sales,stock_value_retail,profit_status,conversion_rate,sales_log,predicted_sales,predicted_profit,image_path,subcategory,category,brand,occasion,size_range,similarity
0,item_000001,2025-01-31,3.0,3736.0,346555.0,172.0,30383339.0,49100152.0,1.00,60.195342,17.229404,127866.250000,profit,../../../fashion_images/dataset_clean/men_carg...,men_cargos,bottoms,Tommy Hilfiger,casual,S,100.0
1,item_000001,2025-02-01,3.0,356.0,216545.0,96.0,24742335.0,25600992.0,1.00,70.376131,17.024027,56849.851562,profit,../../../fashion_images/dataset_clean/formal_s...,formal_shirts,tops,HnM,party,S,100.0
2,item_000001,2025-02-02,0.0,0.0,216545.0,96.0,0.0,25600992.0,1.00,0.000000,0.000000,104502.617188,profit,../../../fashion_images/dataset_clean/formal_s...,formal_shirts,tops,HnM,party,XL,100.0
3,item_000001,2025-02-03,2.0,1044.0,284314.0,49.0,17303367.0,14108619.0,1.00,6.638782,16.666412,32212.800781,profit,../../../fashion_images/dataset_clean/printed_...,printed_tshirts,tops,Tommy Hilfiger,office,XXL,100.0
4,item_000001,2025-02-04,2.0,1104.0,285865.0,148.0,18683496.0,41282232.0,1.00,6.484313,16.743151,75313.539062,profit,../../../fashion_images/dataset_clean/formal_s...,formal_shirts,tops,ZARA,casual,L,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298145,item_000890,2025-12-27,0.0,0.0,305113.5,183.0,0.0,61526064.0,0.75,0.000000,0.000000,13406.049805,profit,../../../fashion_images/dataset_clean/printed_...,printed_tshirts,tops,HnM,party,XL,100.0
298146,item_000890,2025-12-28,1.0,361.0,347344.0,58.0,6252192.0,20145952.0,1.00,4.986150,15.648443,10358.120117,profit,../../../fashion_images/dataset_clean/men_carg...,men_cargos,bottoms,Nike,office,M,100.0
298147,item_000890,2025-12-29,2.0,1571.0,204049.5,68.0,11992583.0,17865164.0,1.00,7.670717,16.299799,96990.546875,profit,../../../fashion_images/dataset_clean/jeans/im...,jeans,bottoms,ZARA,casual,XXL,100.0
298148,item_000890,2025-12-30,0.0,0.0,204049.5,68.0,0.0,17865164.0,1.00,0.000000,0.000000,165973.046875,profit,../../../fashion_images/dataset_clean/men_carg...,men_cargos,bottoms,Nike,office,XXL,100.0


## Index Product via Langchain -> Qdrant Cloud

In [18]:
from langchain_core.documents import Document
from tqdm.auto import tqdm
import numpy as np


def row_to_document(row) -> Document:
    content = (
        f"{row.get('brand', '')}"
        f"{row.get('category', '')}"
        f"{row.get('occasion', '')}"
        f"{row.get('size_range', '')}"
        f"{row.get('subcategory', '')}"
        f"{row.get('sales', '')}"
        f"{row.get('stock_value_retail', '')}"
        f"{row.get('price', '')}"
    )

    # Convert ALL numpy types -> native Python types
    def safe(v):
        if isinstance(v, Path): return str(v)
        if isinstance(v, np.integer): return int(v)
        if isinstance(v, np.floating): return float(v)
        if isinstance(v, np.bool_): return bool(v)
        if pd.isna(v): return ""
        if v is None: return ""
        return str(v)

    metadata = {k: safe(v) for k, v in row.to_dict().items()}
    return Document(page_content=content, metadata=metadata)

if need_index_products:
    display(Markdown("⏳ **Indexing products → Qdrant Cloud...**"))

    docs = [row_to_document(row) for _, row in df_merged.iterrows()]

    # Batch add to avoid rate limits
    BATCH = 64
    with tqdm(total=len(docs), desc="📦 Indexing products", unit="doc") as pbar:
        for start in range(0, len(docs), BATCH):
            batch = docs[start:start + BATCH]
            product_vectorstore.add_documents(batch)
            pbar.update(len(batch))

    display(HTML(f"""
    <div style='border:1px solid #4CAF50;border-radius:8px;padding:10px;background:#f9fff9'>
        ✅ <b>{len(docs)} product documents</b> indexed into <code>{COLLECTION}</code>
    </div>
    """))
else:
    display(Markdown("✅ Products already indexed — skipped"))

✅ Products already indexed — skipped

## Indef PDF

In [23]:
from pypdf import PdfReader

if need_index_pdf:
    PDF_DIR = Path("../../../../pipelines/ai_engineer/docs/RAG_Analysis_Report.pdf")
    pdf_docs = []

    reader = PdfReader(str(PDF_DIR))
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and len(text.strip()) > 50:
            pdf_docs.append(Document(
                page_content=text.strip(),
                metadata={
                    "source": PDF_DIR.name,
                    "page": page_num + 1
                }
            ))

    display(Markdown(f"📄 **{len(pdf_docs)} page documents** extracted"))

    BATCH = 32
    with tqdm(total=len(pdf_docs), desc="📄 Indexing PDF pages", unit="page") as pbar:
        for start in range(0, len(pdf_docs), BATCH):
            batch = pdf_docs[start:start + BATCH]
            pdf_vectorstore.add_documents(batch)
            pbar.update(len(batch))
            
    display(HTML(f"""
        <div style='border:1px solid #4CAF50;border-radius:8px;padding:10px;background:#f9fff9'>
            ✅ <b>{len(pdf_docs)} PDF pages</b> indexed into <code>{PDF_COLLECTION}</code>
        </div>
        """))
else:
    display(Markdown("✅ PDF reports already indexed — skipped"))

✅ PDF reports already indexed — skipped

In [24]:
path = Path("../../../../pipelines/ai_engineer/docs/RAG_Analysis_Report.pdf")

if path.exists():
    display(HTML(f"""
    <div style='border:1px solid #4CAF50;border-radius:8px;padding:10px;background:#f9fff9'>
        ✅ PDF file found at <code>{path}</code> — ready for indexing
    </div>
    """))
else:
    display(HTML(f"""
    <div style='border:1px solid #f44336;border-radius:8px;padding:10px;background:#ffebee'>
        ❌ PDF file not found at <code>{path}</code> — cannot index
    </div>
    """))

## Langchain Retriever + RAG Chain -> Models & Finetuning LLM Deployment
* ### MLflow -> Deploy to MLflow as model provider server
* ### Databaricks -> Maintenance models power

### To ensure deployment we track and logging for experiment & metrics
* ### MLflow tracking uri -> Server UI for MLflow
* ### MLflow log metrics params -> metrics experiment logging

## LLM -> Experiment

In [29]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from tqdm.auto import tqdm
from IPython.display import display, HTML, Markdown

LLM_MODEL = "llama-3.3-70b-versatile"

steps = ["groq_client", "test"]

with tqdm(steps, desc="🔧 Setting up LLM", unit="step", ncols=70) as pbar:

    # Step 1: Create Groq Client
    pbar.set_description("🌐 Connecting to Groq API")
    llm = ChatGroq(
        model=LLM_MODEL,
        groq_api_key=os.getenv("GROQ_API_KEY"),
        max_tokens=512,
        temperature=0.7,
    )
    pbar.update(1)

    # Step 2: Test inference
    pbar.set_description("🤖 Running test inference")
    try:
        response = llm.invoke([
            HumanMessage(content="What's role AI/ML Engineer and Data Scientist in fashion industry? How they collaborate each other to ensure business success?")
        ])
        response_text = response.content
        pbar.update(1)

        display(HTML(f"""
        <div style='border:1px solid #4CAF50;border-radius:8px;padding:12px;background:#f9fff9'>
            ✅ <b>LLM ready</b><br>
            🤗 Model          : <code>{LLM_MODEL}</code><br>
            ☁️ Mode           : <b>Full cloud — no local CPU/GPU</b><br>
            🕒 Max Tokens     : <b>512</b><br>
            🌡️ Temperature    : <b>0.7</b><br>
            🔁 Rep. Penalty   : <b>1.1</b>
        </div>
        """))
        display(Markdown(f"**LLM Response:** {response_text}"))

    except Exception as e:
        pbar.update(1)
        display(HTML(f"<div style='color:red;padding:10px'>❌ LLM failed: {e}</div>"))
        raise e

🤖 Running test inference: 100%|██████| 2/2 [00:01<00:00,  1.17step/s]

**LLM Response:** In the fashion industry, AI/ML Engineers and Data Scientists play crucial roles in driving business success by leveraging data-driven insights, predictive analytics, and machine learning capabilities. Here's an overview of their roles and how they collaborate:

**AI/ML Engineer in Fashion Industry:**

1. **Develop predictive models**: Build and deploy machine learning models to forecast demand, predict trends, and optimize inventory management.
2. **Image and video analysis**: Use computer vision techniques to analyze images and videos of fashion products, enabling tasks like object detection, image classification, and style transfer.
3. **Natural Language Processing (NLP)**: Develop chatbots, sentiment analysis tools, and text classification models to analyze customer feedback, reviews, and social media conversations.
4. **Recommendation systems**: Design and implement personalized recommendation engines to suggest products to customers based on their preferences, behavior, and purchase history.
5. **Supply chain optimization**: Use machine learning to optimize supply chain operations, such as predicting lead times, managing inventory, and streamlining logistics.

**Data Scientist in Fashion Industry:**

1. **Data analysis and insights**: Analyze large datasets to identify trends, patterns, and correlations, providing insights on customer behavior, market trends, and business performance.
2. **Data visualization**: Create interactive dashboards and reports to communicate complex data insights to stakeholders, facilitating data-driven decision-making.
3. **Customer segmentation**: Develop customer segmentation models to identify high-value customer groups, enabling targeted marketing campaigns and personalized experiences.
4. **Market research and trend analysis**: Conduct market research and analyze trends to inform product development, pricing strategies, and marketing campaigns.
5. **A/B testing and experimentation**: Design and execute A/B tests to measure the effectiveness of different marketing strategies, product features, and user experiences.

**Collaboration between AI/ML Engineer and Data Scientist:**

1. **Joint problem definition**: Collaborate to define business problems and identify opportunities for AI/ML applications.
2. **Data preparation**: Work together to collect, preprocess, and integrate data from various sources, ensuring high-quality data for modeling and analysis.
3. **Model development and deployment**: AI/ML Engineers develop and deploy models, while Data Scientists provide input on data quality, feature engineering, and model interpretability.
4. **Model evaluation and refinement**: Collaborate to evaluate model performance, identify areas for improvement, and refine models to ensure they meet business objectives.
5. **Insight generation and storytelling**: Data Scientists generate insights from data analysis, while AI/ML Engineers provide technical

🤖 Running test inference: 100%|██████| 2/2 [00:01<00:00,  1.17step/s]


## LLM -> Implementation

In [30]:
from langchain_core.prompts import  PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Retrievers from both vectorstores
product_retriever = product_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

pdf_retriever = pdf_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Prompt template for LLM
RAG_PROMPT = PromptTemplate.from_template("""
<s>[INST] You are a senior machine learning engineer specializing in the fashion industry where works in ecommerce companies.
Use the product catalog and report insights below to answer properly.
                                          

### Product Catalog:
{products}

### Report Insights:
{pdf_context}

### Question:
{question} / [/INST]                                          
""")

def format_products(docs: list) -> str:
    return "\n".join([
        f"- {d.metadata.get('brand', 'unknown brand')} | "
        f" {d.metadata.get('category', 'unknown category')} | "
        f"Rp.{d.metadata.get('price', 'unknown price')}"
        for d in docs
    ])

def format_pdf(docs: list) -> str:
    return "\n\n".join([
        f"[{d.metadata.get('source', 'unknown source')} p.{d.metadata.get('page', 'unknown page')}]:"
        f"{d.page_content[:300]}"
        for d in docs
    ])

# RAG Chain
rag_chain = (
    {
        "products": product_retriever | RunnableLambda(format_products),
        "pdf_context": pdf_retriever | RunnableLambda(format_pdf),
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

display(Markdown("✅ **RAG chain built** — LangChain + HuggingFaceEndpoint + Qdrant Cloud"))

✅ **RAG chain built** — LangChain + HuggingFaceEndpoint + Qdrant Cloud

## Generate to mlflow

In [36]:
import os
import mlflow
import time
from dotenv import load_dotenv
from pathlib import Path
from groq import Groq
from IPython.display import display, HTML, Markdown
from mlflow import MlflowClient

# Load environment variables
env_path = Path.cwd().resolve().parents[3] / ".env"
load_dotenv(dotenv_path=env_path)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

time.sleep(2)
mlflow.set_tracking_uri("http://localhost:5007")

try:
    mlflow.set_experiment("groq_inference")

    with mlflow.start_run(run_name="groq_test") as run:
        display(Markdown("## 🚀 Starting RAG + Groq Inference Pipeline"))
        
        question = "What are the top fashion trends in our product catalog and how should we optimize recommendations?"
        display(Markdown(f"### Question: {question}"))

        # ✅ Step 1: Invoke RAG chain
        display(Markdown("⏳ **Step 1:** Retrieving context from Qdrant..."))
        rag_response = rag_chain.invoke(question)
        display(Markdown(f"✅ Retrieved {len(rag_response)} characters of context"))

        # ✅ Step 2: Send to Groq API for inference
        display(Markdown("⏳ **Step 2:** Sending to Groq LLM for inference..."))
        client = Groq()
        message = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": rag_response
            }],
            temperature=0.7
        )
        
        response_text = message.choices[0].message.content
        token_count = len(response_text.split())
        
        display(Markdown(f"✅ Generated {token_count} tokens"))

        # ✅ Step 3: Log parameters and metrics to MLflow
        display(Markdown("⏳ **Step 3:** Logging to MLflow..."))
        mlflow.log_param("model", "llama-3.3-70b-versatile")
        mlflow.log_param("temperature", 0.7)
        mlflow.log_param("question", question)
        mlflow.log_param("retrieval_k_products", 5)
        mlflow.log_param("retrieval_k_pdf", 3)

        mlflow.log_metric("response_length", len(response_text))
        mlflow.log_metric("token_count", token_count)
        mlflow.log_metric("context_length", len(rag_response))
        
        # ✅ Log full response as artifact
        mlflow.log_text(response_text, artifact_file="groq_response.txt")
        
        # ✅ NEW: Log context as artifact for traceability
        mlflow.log_text(rag_response, artifact_file="rag_context.txt")
        
        display(Markdown("✅ Successfully logged to MLflow"))

        # ✅ Step 4: Display results
        display(HTML(f"""
        <div style='border:2px solid #4CAF50;padding:15px;border-radius:10px;background:#f9fff9'>
            <h3>✅ MLflow Run Completed Successfully</h3>
            <p><b>Status:</b> ✅ Successfully logged to MLflow server</p>
            <p><b>Run ID:</b> <code>{run.info.run_id}</code></p>
            <p><b>Question:</b> <code>{question}</code></p>
            <p><b>Metrics:</b></p>
            <ul>
                <li>Response Length: <b>{len(response_text)}</b> chars</li>
                <li>Token Count: <b>{token_count}</b></li>
                <li>Context Length: <b>{len(rag_response)}</b> chars</li>
            </ul>
        </div>
        """))
        
        # ✅ Display full response
        display(Markdown("### 📋 Full Response from Groq:"))
        display(Markdown(f"{response_text}"))
        
        # ✅ Display MLflow links
        run_id = run.info.run_id
        exp_id = run.info.experiment_id
        
        display(HTML(f"""
        <p style='text-align:center;margin-top:20px'>
            <a href='http://localhost:5007/#/experiments/{exp_id}/runs/{run_id}' 
               target='_blank' 
               style='background:#4CAF50;color:white;padding:12px 25px;border-radius:6px;text-decoration:none;font-weight:bold;font-size:14px;display:inline-block;margin-right:10px'>
               📊 View Run Details
            </a>
            <a href='http://localhost:5007/#/experiments/{exp_id}' 
               target='_blank' 
               style='background:#2196F3;color:white;padding:12px 25px;border-radius:6px;text-decoration:none;font-weight:bold;font-size:14px;display:inline-block'>
               📈 View Experiment
            </a>
        </p>
        """))
        
except Exception as e:
    display(HTML(f"""
    <div style='border:2px solid #f44336;padding:15px;border-radius:10px;background:#ffebee'>
        <h3>❌ Error Occurred</h3>
        <p><b>Error Type:</b> {type(e).__name__}</p>
        <p><b>Error Message:</b> {str(e)}</p>
    </div>
    """))
    import traceback
    traceback.print_exc()

## 🚀 Starting RAG + Groq Inference Pipeline

### Question: What are the top fashion trends in our product catalog and how should we optimize recommendations?

⏳ **Step 1:** Retrieving context from Qdrant...

✅ Retrieved 2816 characters of context

⏳ **Step 2:** Sending to Groq LLM for inference...

...

⏳ **Step 3:** Logging to MLflow...

✅ Successfully logged to MLflow

### 📋 Full Response from Groq:

Your analysis of the product catalog and suggestions for optimizing recommendations are well-reasoned and comprehensive. Based on the limited dataset, you've identified potential trends, such as the popularity of Adidas bottoms and mid-range pricing, which can inform your recommendation strategy.

Your proposed methods for optimizing recommendations, including collaborative filtering, content-based filtering, hybrid approaches, personalization, and diversification, are all relevant and effective techniques for improving customer engagement and sales.

The additional suggestions for increasing customer engagement and optimizing recommendations, such as analyzing customer feedback, monitoring sales trends, conducting A/B testing, and using data visualization, are also excellent ideas that can help refine and improve the recommendation strategy.

To further build on your analysis, here are a few potential next steps:

1. **Collect more data**: As you mentioned, having more data on sales figures, customer demographics, and preferences would be invaluable in refining the recommendation strategy.
2. **Explore customer segmentation**: Segmenting customers based on their behavior, preferences, and demographics can help create more targeted and effective recommendations.
3. **Consider external factors**: Keep an eye on external factors, such as seasonal trends, fashion trends, and social media influence, which can impact customer behavior and preferences.
4. **Continuously evaluate and refine**: Regularly evaluate the performance of the recommendation strategy and refine it as needed to ensure it remains effective and aligned with customer needs.

Overall, your analysis and suggestions provide a solid foundation for optimizing recommendations and improving customer engagement. By continuing to collect and analyze data, refine the recommendation strategy, and stay attuned to customer needs and preferences, you can create a highly effective and personalized shopping experience for your customers.

🏃 View run groq_test at: http://localhost:5007/#/experiments/219342459421682435/runs/230b5a8984dd4ed9bd4dff4ce20a7749
🧪 View experiment at: http://localhost:5007/#/experiments/219342459421682435


In [ ]:

from IPython.display import HTML, display
import mlflow

# ✅ Set tracking URI
mlflow.set_tracking_uri("http://localhost:5007")

# Get the experiment and latest run
experiment = mlflow.get_experiment_by_name("groq_inference")

if experiment:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id], 
        max_results=1,
        order_by=["start_time DESC"]
    )
    
    if len(runs) > 0:
        latest_run = runs.iloc[0]
        run_id = latest_run.run_id
        exp_id = experiment.experiment_id
        
        # ✅ Display clickable link (NOT IFrame)
        display(HTML(f"""
        <div style='border:3px solid #4CAF50;padding:25px;border-radius:10px;background:#f0f9f0;text-align:center'>
            <h2 style='color:#2e7d32;margin:0 0 15px 0'>✅ MLflow Live Connection</h2>
            <p style='color:#666;font-size:14px;margin:10px 0'><b>Experiment:</b> groq_inference</p>
            <p style='color:#666;font-size:14px;margin:10px 0'><b>Run ID:</b> <code>{run_id}</code></p>
            <p style='margin:20px 0 0 0'>
                <a href='http://localhost:5007/#/experiments/{exp_id}/runs/{run_id}' 
                   target='_blank' 
                   style='background:#4CAF50;color:white;padding:15px 35px;border-radius:6px;text-decoration:none;font-weight:bold;font-size:16px;display:inline-block'>
                   📊 Open MLflow Dashboard
                </a>
            </p>
        </div>
        """))
        print(f"✅ Run ID: {run_id}")
        print(f"✅ View at: http://localhost:5007/#/experiments/{exp_id}/runs/{run_id}")
    else:
        print("❌ No runs found")
else:
    print("❌ Experiment not found")

✅ Run ID: f1cb854d9ba84b7eb8afcb926b616d8e
✅ View at: http://localhost:5007/#/experiments/219342459421682435/runs/f1cb854d9ba84b7eb8afcb926b616d8e


## Databricks & MLflow deployment